# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wajiha-Waqar/FlyRankInternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis

One row represents one content page for one day for one client from the
fact_content_daily_performance table.

For this assignment I will use only the March 2026 partition
(month = 2026-03) as a mid-panel snapshot.

This allows me to explore the data without using the final month
(June 2026), which is reserved for future outcome evaluation.

The goal is to rank pages that should be reviewed first for content
refresh.

Output:
A ranked review queue for editors.

Prediction target:
Pages likely to require content review based on observable search
performance signals.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features

• gsc_impressions
• gsc_clicks
• gsc_avg_position
• ga4_sessions
• content_age_days

These are observable signals that are available before a review decision
is made.

Label / Proxy

Future decline or the starter proxy
(is_declining_label).

This is what the model predicts and must never be used as a feature.

Context

client_hash_id
content_hash_id
report_date

These identify rows and are used for grouping or joining, not for
prediction.

Excluded

trend_direction
trend_pct

Reason:
These are directly related to the target and would cause target leakage.

Also excluded:

raw URLs
client names
private queries

These are excluded for privacy and because they are not available in the
public release.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
!pip install duckdb pandas pyarrow hf_xet

In [20]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(f"""
CREATE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [21]:
con.sql("""

SELECT
report_date,
client_hash_id,
content_hash_id,
COUNT(*) AS duplicates

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

GROUP BY
report_date,
client_hash_id,
content_hash_id

HAVING COUNT(*)>1

LIMIT 10;

""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicates


In [22]:
con.sql("""

SELECT

COUNT(*) AS total_rows,

MIN(report_date) AS first_day,

MAX(report_date) AS last_day

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);

""").df()

,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


In [23]:
con.sql("""

SELECT

COUNT(*) AS usable_rows

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE ga4_data_available IS TRUE;

""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,usable_rows
0,413966


#Five Features
| Feature          | Available when?                                                                            |
| ---------------- | ------------------------------------------------------------------------------------------ |
| gsc_impressions  | Available before the review decision because it comes from historical Search Console data. |
| gsc_clicks       | Available before the review decision because it records past user behavior.                |
| gsc_avg_position | Known before ranking pages since it reflects historical search rankings.                   |
| ga4_sessions     | Available before the decision because it summarizes previous traffic.                      |
| content_age_days | Known at all times from content metadata.                                                  |


#Leakage experiment

I deliberately added trend_direction
(which directly defines is_declining_label)
as a feature.

The model performance became unrealistically high because the feature
already contains the answer.

I removed it immediately.

Final feature set excludes

trend_direction

trend_pct

is_declining_label

to avoid target leakage.

In [24]:
import pandas as pd

df = pd.DataFrame({
    "trend_direction":["down","up","down"],
})

df["is_declining_label"] = (
    df["trend_direction"]=="down"
).astype(int)

df

,trend_direction,is_declining_label
0,down,1
1,up,0
2,down,1


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice has several limitations.

Only one month (March 2026) is used.

Client histories are uneven because tracking started at different times.

Rows before GA4 tracking may contain only Search Console data.

This analysis cannot prove that refreshing a page causes traffic
improvements.

It provides decision support, not causal evidence.

The warehouse excludes private URLs, client names and search queries, so
recommendations remain privacy-safe.

In [25]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.